In [ ]:
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, roc_auc_score,
    f1_score, accuracy_score,
    mean_absolute_error, mean_squared_error,
    roc_curve
)
from google.colab import drive

drive.mount('/content/drive')

BASE          = 'YOUR_DRIVE_PATH'
RESULT_FOLDER = f'{BASE}/results'
EVAL_FOLDER   = f'{BASE}/results/evaluation'
os.makedirs(EVAL_FOLDER, exist_ok=True)

In [ ]:
# LLM Baseline
with open(f'{RESULT_FOLDER}/baseline_llm.json') as f:
    llm_results = json.load(f)

# Data Strategy Results (07f)
with open(
    f'{RESULT_FOLDER}/data_strategy_results.json'
) as f:
    strategy_results = json.load(f)

# Per-domain Results (07d)
with open(
    f'{RESULT_FOLDER}/per_domain_results.json'
) as f:
    per_domain_results = json.load(f)

# model v1 (07_train_model)
v1_results = {
    'resnet50_cls': {
        'sensitivity': 0.778, 'specificity': 0.843,
        'auc': 0.870, 'f1': 0.712,
        'name': 'ResNet50 + CLS'
    },
    'resnet50_reg': {
        'sensitivity': 0.886, 'specificity': 0.616,
        'auc': 0.838, 'f1': 0.615,
        'name': 'ResNet50 + REG'
    },
    'vit_cls': {
        'sensitivity': 0.713, 'specificity': 0.907,
        'auc': 0.859, 'f1': 0.730,
        'name': 'ViT + CLS'
    },
    'vit_reg': {
        'sensitivity': 0.719, 'specificity': 0.875,
        'auc': 0.866, 'f1': 0.704,
        'name': 'ViT + REG'
    },
}

# No Rotation (07e)
no_rot_results = {
    'resnet50_cls': {
        'sensitivity': 0.766, 'specificity': 0.877,
        'auc': 0.868, 'name': 'ResNet50 CLS (No Rot)'
    },
    'resnet50_reg': {
        'sensitivity': 0.886, 'specificity': 0.690,
        'auc': 0.873, 'name': 'ResNet50 REG (No Rot)'
    },
    'vit_cls': {
        'sensitivity': 0.826, 'specificity': 0.708,
        'auc': 0.845, 'name': 'ViT CLS (No Rot)'
    },
    'vit_reg': {
        'sensitivity': 0.850, 'specificity': 0.708,
        'auc': 0.868, 'name': 'ViT REG (No Rot)'
    },
}

In [ ]:
fix_rot_path = f'{RESULT_FOLDER}/07h_final_results.json'

if os.path.exists(fix_rot_path):
    with open(fix_rot_path) as f:
        fix_rot_results = json.load(f)

    fix_A = fix_rot_results.get('A', {})
    fix_B = fix_rot_results.get('B', {})
else:
    print('ไม่พบผล 07h')
    fix_A = {}
    fix_B = {}

In [ ]:
print('='*75)
print('COMPLETE MODEL COMPARISON')
print('='*75)
print(f'{"Model":<35} {"Sens":>7} {"Spec":>7} '
      f'{"AUC":>7} {"Pass?":>7}')
print('='*75)

# LLM Baseline
print('\n── LLM Baseline ──')
for cutoff, sens, spec in [
    (6, 0.731, 0.975),
    (7, 0.909, 0.958)
]:
    passed = '✅' if sens >= 0.88 and spec >= 0.82 \
             else '⚠️'
    print(f'  {"LLM (cutoff="+str(cutoff)+")":<33} '
          f'{sens:>7.3f} {spec:>7.3f} '
          f'{"0.944":>7} {passed:>7}')

# v1 Models (with rotation augment)
print('\n── DL Models v1 (With Rotation Augment) ──')
for key, r in v1_results.items():
    passed = '✅' if (r['sensitivity'] >= 0.88 and
                      r['specificity'] >= 0.82) else '⚠️'
    print(f'  {r["name"]:<33} '
          f'{r["sensitivity"]:>7.3f} '
          f'{r["specificity"]:>7.3f} '
          f'{r["auc"]:>7.3f} {passed:>7}')

# No Rotation Models (07e)
print('\n── DL Models (No Rotation Augment) ──')
for key, r in no_rot_results.items():
    passed = '✅' if (r['sensitivity'] >= 0.88 and
                      r['specificity'] >= 0.82) else '⚠️'
    print(f'  {r["name"]:<33} '
          f'{r["sensitivity"]:>7.3f} '
          f'{r["specificity"]:>7.3f} '
          f'{r["auc"]:>7.3f} {passed:>7}')

# Per-domain Model (07d)
print('\n── Per-Domain Training (07d) ──')
cutoffs_07d = [
    (6, 0.605, 0.924, 0.869),
    (7, 0.713, 0.856, 0.869),
    (8, 0.862, 0.759, 0.869),
]
for cutoff, sens, spec, auc in cutoffs_07d:
    passed = '✅' if sens >= 0.88 and spec >= 0.82 \
             else '⚠️'
    print(f'  {"Per-Domain (cutoff="+str(cutoff)+")":<33} '
          f'{sens:>7.3f} {spec:>7.3f} '
          f'{auc:>7.3f} {passed:>7}')

print('\n── Fix Rotation (07h) ──')
fix_models = {
    'ResNet50 REG (Fix+NoRot)' : fix_A.get(
        'resnet50_reg', {}),
    'ResNet50 CLS (Fix+NoRot)' : fix_A.get(
        'resnet50_cls', {}),
    'ViT REG (Fix+NoRot)'      : fix_A.get(
        'vit_reg', {}),
    'ViT CLS (Fix+NoRot)'      : fix_A.get(
        'vit_cls', {}),
}

for name, r in fix_models.items():
    if not r:
        continue
    sens  = r.get('sensitivity', 0)
    spec  = r.get('specificity', 0)
    auc   = r.get('auc', 0)
    passed = '✅' if (sens >= 0.88 and
                      spec >= 0.82) else '⚠️'
    print(f'  {name:<33} '
          f'{sens:>7.3f} {spec:>7.3f} '
          f'{auc:>7.3f} {passed:>7}')

# Data Strategy (07f)
print('\n── Data Strategy Comparison (07f) ──')
strategy_names = {
    'A_no_aug_full'      : 'A: No Aug, Full Data',
    'B_no_aug_undersample': 'B: No Aug, Undersample',
    'C_aug_rotation'     : 'C: Aug + Rotation',
    'D_aug_no_rotation'  : 'D: Aug, No Rotation',
}


for key, name in strategy_names.items():
    if key in strategy_results:
        r      = strategy_results[key]
        passed = '✅' if (r['sensitivity'] >= 0.88 and
                          r['specificity'] >= 0.82) \
                 else '⚠️'
        print(f'  {name:<33} '
              f'{r["sensitivity"]:>7.3f} '
              f'{r["specificity"]:>7.3f} '
              f'{r["auc"]:>7.3f} {passed:>7}')

print('='*75)
print(f'{"Target":<35} {"≥0.880":>7} '
      f'{"≥0.820":>7} {"≥0.910":>7}')

In [ ]:
print('='*65)
print('SUMMARY: Best Choice per Category')
print('='*65)

print('''
┌─────────────────────────────────────────────────────────┐
│  Category          │  Best Choice       │  Why          │
├─────────────────────────────────────────────────────────┤
│  Overall Model     │  LLM (cutoff=7)    │  Sens=0.909   │
│                    │                    │  Spec=0.958✅ │
├─────────────────────────────────────────────────────────┤
│  Best DL Model     │  ResNet50 + REG    │  Sens=0.886 ✅│
│                    │                    │  AUC สูงสุด     │
├─────────────────────────────────────────────────────────┤
│  Loss Function     │  Regression (MSE)  │  Sens สูงกว่า   │
│                    │                    │  CLS เสมอ     │
├─────────────────────────────────────────────────────────┤
│  Augmentation      │  No Augment        │  Stable กว่า   │
│                    │  (Strategy B)      │  ไม่ overfit   │
├─────────────────────────────────────────────────────────┤
│  Rotation          │  ไม่หมุน             │  เข็มนาฬิกา     │
│                    │                    │  ไม่ผิดทิศ      │
├─────────────────────────────────────────────────────────┤
│  Training          │  Multi-domain      │  ดีกว่า         │
│                    │  (ทุก domain รวม)   │  Per-domain   │
└─────────────────────────────────────────────────────────┘
''')

print('''
Key Findings:
1. LLM baseline ดีที่สุด แต่มี circular evaluation problem
   เพราะ test labels มาจาก LLM เช่นกัน

2. ResNet50 + MSE ดีที่สุดในบรรดา DL models
   Sensitivity = 0.886 ✅ แต่ Specificity ต่ำ

3. Regression loss ดีกว่า Classification ทุกกรณี
   เพราะ CDT score เป็น ordinal (0→1→2 มีลำดับ)

4. Rotation augment ทำให้แย่ลง
   เพราะเข็มนาฬิกาที่หมุนแล้ว label ผิดทิศ

5. Data augmentation ไม่ได้ช่วยมากนัก
   อาจเพราะ label มาจาก LLM มี noise สูง
''')

In [ ]:
print('='*65)
print('PER-DOMAIN PERFORMANCE ANALYSIS')
print('='*65)

domain_data = {
    'A: digit_count'    : {'acc': 0.726, 'mae': 0.289,
                            'mse': 0.319, 'f1': 0.537},
    'B: worst_quadrant' : {'acc': 0.529, 'mae': 0.481,
                            'mse': 0.501, 'f1': 0.430},
    'C: spatial'        : {'acc': 0.693, 'mae': 0.311,
                            'mse': 0.317, 'f1': 0.558},
    'D: hands_present'  : {'acc': 0.679, 'mae': 0.336,
                            'mse': 0.366, 'f1': 0.502},
    'E: hands_placement': {'acc': 0.636, 'mae': 0.392,
                            'mse': 0.449, 'f1': 0.496},
}

print(f'\n{"Domain":<25} {"Acc":>7} {"MAE":>7} '
      f'{"MSE":>7} {"F1":>7} {"Difficulty"}')
print('-'*70)

for name, d in domain_data.items():
    if d['acc'] >= 0.70:
        difficulty = '🟢 ง่าย'
    elif d['acc'] >= 0.60:
        difficulty = '🟡 ปานกลาง'
    else:
        difficulty = '🔴 ยาก'
    print(f'{name:<25} {d["acc"]:>7.3f} {d["mae"]:>7.3f} '
          f'{d["mse"]:>7.3f} {d["f1"]:>7.3f} {difficulty}')

print('\n')
print('Per-Class Accuracy (ResNet50 REG — No Rotation):')
print('-'*70)

class_data = {
    'A: digit_count': {
        0: (36, 77), 1: (59, 76), 2: (212, 446)
    },
    'B: worst_quadrant': {
        0: (73, 138), 1: (253, 287), 2: (6, 174)
    },
    'C: spatial': {
        0: (34, 69), 1: (96, 128), 2: (231, 402)
    },
    'D: hands_present': {
        0: (26, 87), 1: (68, 92), 2: (270, 420)
    },
    'E: hands_placement': {
        0: (63, 133), 1: (42, 53), 2: (160, 413)
    },
}

print(f'\n{"Domain":<25} {"Class 0":>10} '
      f'{"Class 1":>10} {"Class 2":>10}')
print('-'*60)

for domain, classes in class_data.items():
    row = f'{domain:<25}'
    for c in [0, 1, 2]:
        correct, total = classes[c]
        acc   = correct/total if total > 0 else 0
        mark  = '⚠️' if acc < 0.5 else ''
        row  += f'  {acc:.3f}{mark:>3}'
    print(row)

print('''
Class Analysis Insights:
- Class 0 (คะแนน 0): ทายผิดบ่อย → รูปที่ขาดมากๆ น้อย
- Class 1 (คะแนน 1): ทายดีที่สุดเสมอ → model bias กลาง
- Class 2 (คะแนน 2): Domain B แย่มาก (acc=0.034)
  → model ไม่เข้าใจ worst_quadrant ระดับดี
''')

In [ ]:
print('='*65)
print('ERROR ANALYSIS')
print('='*65)

print('''
ประเภทของ Error:

TP (True Positive)  = ทาย dementia ถูก
TN (True Negative)  = ทาย ปกติ ถูก
FP (False Positive) = ทาย dementia แต่จริงๆ ปกติ ← false alarm
FN (False Negative) = ทาย ปกติ แต่จริงๆ dementia ← อันตรายมาก!
''')

# Typical error counts จาก ResNet50 REG v1
error_summary = {
    'ResNet50 REG v1': {
        'TP': 73, 'TN': 429, 'FP': 50, 'FN': 47
    },
    'ViT REG v1': {
        'TP': 59, 'TN': 461, 'FP': 18, 'FN': 61
    },
    'LLM (cutoff=7)': {
        'TP': 75, 'TN': 503, 'FP': 21, 'FN': 8
    }
}

print(f'\n{"Model":<25} {"TP":>5} {"TN":>5} '
      f'{"FP":>5} {"FN":>5} {"FN Rate":>10}')
print('-'*60)

for model_name, counts in error_summary.items():
    total_pos = counts['TP'] + counts['FN']
    fn_rate   = counts['FN'] / total_pos \
                if total_pos > 0 else 0
    print(f'{model_name:<25} '
          f'{counts["TP"]:>5} {counts["TN"]:>5} '
          f'{counts["FP"]:>5} {counts["FN"]:>5} '
          f'{fn_rate:>10.3f}')

print('''
FN Rate = สัดส่วนที่พลาด dementia
ยิ่งต่ำยิ่งดี (LLM ดีที่สุด = 0.096)
''')

# หา insight จาก error
print('='*65)
print('INSIGHTS FROM ERROR ANALYSIS')
print('='*65)

print('''
1. Domain B (worst_quadrant) คือปัญหาหลัก
   ─────────────────────────────────────
   • Class 2 accuracy = 0.034 (แย่มากที่สุด!)
   • Model ทาย Class 1 แทน Class 2 เกือบทั้งหมด
   • สาเหตุ: "worst quadrant" ต้องการ spatial reasoning
     ที่ ResNet/ViT ยังทำได้ไม่ดีพอ
   • แนวทางแก้: ใช้ attention mechanism หรือ
     crop เฉพาะ quadrant มา train แยก

2. Model Bias ไปทาง Class 1 (กลางๆ)
   ─────────────────────────────────────
   • ทุก domain มีปัญหาเดียวกัน
   • Class 0 และ Class 2 accuracy ต่ำกว่า Class 1
   • สาเหตุ: Class 1 มีจำนวนมากที่สุดใน dataset
   • แนวทางแก้: ใช้ Focal Loss หรือ
     class weight per class (ไม่ใช่แค่ dementia/normal)

3. FN Cases มักเป็นรูป dementia ที่คะแนนใกล้ cutoff
   ─────────────────────────────────────
   • รูปที่ true_total = 5 (ใกล้ cutoff=6 มาก)
   • Model ทาย 6-7 แทน
   • สาเหตุ: borderline cases ยากมาก
   • แนวทางแก้: เพิ่ม weight สำหรับ borderline cases

4. Label Quality เป็นปัญหาหลัก
   ─────────────────────────────────────
   • LLM label มี noise โดยเฉพาะ Domain B
   • Model เรียนรู้ pattern ที่ผิดพลาด
   • แนวทางแก้: Human expert label
     อย่างน้อยสำหรับ Domain B

5. Data น้อยเกินไป
   ─────────────────────────────────────
   • Dementia จริงๆ แค่ 686 รูป
   • แม้ augment แล้วยังไม่พอ
   • แนวทางแก้: เก็บ data เพิ่ม
     หรือ transfer learning จาก CDT dataset อื่น
''')

In [ ]:
fig = plt.figure(figsize=(20, 24))
gs  = gridspec.GridSpec(4, 3, figure=fig,
                         hspace=0.4, wspace=0.3)

# ── กราฟ 1: Model Comparison — Sensitivity ──
ax1 = fig.add_subplot(gs[0, 0])

all_models_data = {
    'LLM\n(cut=7)'    : (0.909, 0.958),
    'ResNet\nCLS v1'  : (0.778, 0.843),
    'ResNet\nREG v1'  : (0.886, 0.616),
    'ViT\nCLS v1'     : (0.713, 0.907),
    'ViT\nREG v1'     : (0.719, 0.875),
    'ResNet\nCLS NoR' : (0.766, 0.877),
    'ResNet\nREG NoR' : (0.886, 0.690),
    'ViT\nCLS NoR'    : (0.826, 0.708),
    'ViT\nREG NoR'    : (0.850, 0.708),
}

names = list(all_models_data.keys())
sens  = [v[0] for v in all_models_data.values()]
spec  = [v[1] for v in all_models_data.values()]
colors_bar = ['gold' if s >= 0.88 else 'steelblue'
              for s in sens]

bars = ax1.bar(names, sens, color=colors_bar,
               edgecolor='white')
ax1.axhline(y=0.88, color='red', linestyle='--',
            label='Target 0.88')
ax1.set_title('Sensitivity — All Models', fontsize=11)
ax1.set_ylim(0, 1.1)
ax1.set_xticklabels(names, fontsize=7)
ax1.legend(fontsize=8)
for bar, val in zip(bars, sens):
    ax1.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.01,
             f'{val:.2f}', ha='center', fontsize=7)

# ── กราฟ 2: Model Comparison — Specificity ──
ax2 = fig.add_subplot(gs[0, 1])
colors_bar2 = ['gold' if s >= 0.82 else 'coral'
               for s in spec]
bars2 = ax2.bar(names, spec, color=colors_bar2,
                edgecolor='white')
ax2.axhline(y=0.82, color='red', linestyle='--',
            label='Target 0.82')
ax2.set_title('Specificity — All Models', fontsize=11)
ax2.set_ylim(0, 1.1)
ax2.set_xticklabels(names, fontsize=7)
ax2.legend(fontsize=8)
for bar, val in zip(bars2, spec):
    ax2.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.01,
             f'{val:.2f}', ha='center', fontsize=7)

# ── กราฟ 3: Sens vs Spec Scatter ──
ax3 = fig.add_subplot(gs[0, 2])
scatter_data = {
    'LLM (cut=7)'     : (0.909, 0.958, 'gold',   '*', 200),
    'ResNet REG v1'   : (0.886, 0.616, 'red',     'o', 100),
    'ResNet CLS v1'   : (0.778, 0.843, 'blue',    'o', 100),
    'ViT CLS v1'      : (0.713, 0.907, 'green',   's', 100),
    'ViT REG v1'      : (0.719, 0.875, 'purple',  's', 100),
    'ResNet REG NoRot': (0.886, 0.690, 'orange',  '^', 100),
    'ViT CLS NoRot'   : (0.826, 0.708, 'cyan',    '^', 100),
}
for name, (s, sp, c, m, sz) in scatter_data.items():
    ax3.scatter(sp, s, color=c, marker=m,
                s=sz, label=name, zorder=5)

ax3.axhline(y=0.88, color='red', linestyle='--',
            alpha=0.5)
ax3.axvline(x=0.82, color='blue', linestyle='--',
            alpha=0.5)
ax3.fill_between([0.82, 1.0], [0.88, 0.88],
                 [1.0, 1.0], alpha=0.1,
                 color='green', label='Target Zone')
ax3.set_xlabel('Specificity')
ax3.set_ylabel('Sensitivity')
ax3.set_title('Sensitivity vs Specificity', fontsize=11)
ax3.set_xlim(0.4, 1.05)
ax3.set_ylim(0.6, 1.05)
ax3.legend(fontsize=6, loc='lower left')

# ── กราฟ 4: Data Strategy Comparison ──
ax4 = fig.add_subplot(gs[1, 0])
strategy_names_short = ['A\nFull', 'B\nUndersample',
                         'C\nAug+Rot', 'D\nAug NoRot']
strategy_keys = ['A_no_aug_full', 'B_no_aug_undersample',
                 'C_aug_rotation', 'D_aug_no_rotation']

s_sens = []
s_spec = []
for key in strategy_keys:
    if key in strategy_results:
        s_sens.append(strategy_results[key]['sensitivity'])
        s_spec.append(strategy_results[key]['specificity'])
    else:
        s_sens.append(0)
        s_spec.append(0)

x     = np.arange(len(strategy_names_short))
width = 0.35
ax4.bar(x - width/2, s_sens, width,
        label='Sensitivity', color='steelblue',
        edgecolor='white')
ax4.bar(x + width/2, s_spec, width,
        label='Specificity', color='coral',
        edgecolor='white')
ax4.axhline(y=0.88, color='steelblue',
            linestyle='--', alpha=0.5)
ax4.axhline(y=0.82, color='coral',
            linestyle='--', alpha=0.5)
ax4.set_xticks(x)
ax4.set_xticklabels(strategy_names_short)
ax4.set_title('Data Strategy Comparison', fontsize=11)
ax4.set_ylim(0, 1.1)
ax4.legend(fontsize=8)

# ── กราฟ 5: Per-Domain Performance ──
ax5 = fig.add_subplot(gs[1, 1])
domain_names  = ['A', 'B', 'C', 'D', 'E']
domain_accs   = [0.726, 0.529, 0.693, 0.679, 0.636]
domain_maes   = [0.289, 0.481, 0.311, 0.336, 0.392]
colors_domain = ['green' if a >= 0.70 else
                 'orange' if a >= 0.60 else
                 'red' for a in domain_accs]

ax5_twin = ax5.twinx()
bars5 = ax5.bar(domain_names, domain_accs,
                color=colors_domain, alpha=0.7,
                edgecolor='white', label='Accuracy')
ax5_twin.plot(domain_names, domain_maes, 'b-o',
              label='MAE', linewidth=2)
ax5.set_title('Per-Domain Performance', fontsize=11)
ax5.set_ylabel('Accuracy', color='gray')
ax5_twin.set_ylabel('MAE', color='blue')
ax5.set_ylim(0, 1)
ax5_twin.set_ylim(0, 1)
ax5.legend(loc='upper left', fontsize=8)
ax5_twin.legend(loc='upper right', fontsize=8)
for bar, val in zip(bars5, domain_accs):
    ax5.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.01,
             f'{val:.2f}', ha='center', fontsize=9)

# ── กราฟ 6: Per-Class Accuracy Heatmap ──
ax6 = fig.add_subplot(gs[1, 2])
class_matrix = np.array([
    [0.468, 0.776, 0.475],  # A
    [0.529, 0.882, 0.034],  # B ← 0.034 แดงสุด!
    [0.493, 0.750, 0.575],  # C
    [0.299, 0.739, 0.643],  # D
    [0.474, 0.792, 0.387],  # E
])
sns.heatmap(
    class_matrix,
    annot=True, fmt='.3f',
    cmap='RdYlGn',
    ax=ax6,
    xticklabels=['Class 0', 'Class 1', 'Class 2'],
    yticklabels=['A', 'B', 'C', 'D', 'E'],
    vmin=0, vmax=1
)
ax6.set_title('Per-Class Accuracy per Domain\n'
              '(ResNet50 REG, No Rotation)',
              fontsize=10)

# ── กราฟ 7: Augmentation Effect ──
ax7 = fig.add_subplot(gs[2, 0])
aug_comparison = {
    'With\nRotation' : (0.886, 0.616),
    'No\nRotation'   : (0.886, 0.690),
    'No\nAugment'    : (0.0,   0.0),
}
# อัปเดตด้วยค่าจริงถ้ามี
if 'B_no_aug_undersample' in strategy_results:
    r = strategy_results['B_no_aug_undersample']
    aug_comparison['No\nAugment'] = (
        r['sensitivity'], r['specificity']
    )

aug_names = list(aug_comparison.keys())
aug_sens  = [v[0] for v in aug_comparison.values()]
aug_spec  = [v[1] for v in aug_comparison.values()]

x     = np.arange(len(aug_names))
width = 0.35
ax7.bar(x - width/2, aug_sens, width,
        label='Sensitivity', color='steelblue',
        edgecolor='white')
ax7.bar(x + width/2, aug_spec, width,
        label='Specificity', color='coral',
        edgecolor='white')
ax7.axhline(y=0.88, color='steelblue',
            linestyle='--', alpha=0.5)
ax7.axhline(y=0.82, color='coral',
            linestyle='--', alpha=0.5)
ax7.set_xticks(x)
ax7.set_xticklabels(aug_names)
ax7.set_title('Effect of Rotation Augmentation\n'
              '(ResNet50 REG)', fontsize=11)
ax7.set_ylim(0, 1.1)
ax7.legend(fontsize=8)

# ── กราฟ 8: Loss Function Comparison ──
ax8 = fig.add_subplot(gs[2, 1])
loss_data = {
    'ResNet\nCLS': (0.778, 0.843),
    'ResNet\nREG': (0.886, 0.616),
    'ViT\nCLS'  : (0.713, 0.907),
    'ViT\nREG'  : (0.719, 0.875),
}
loss_names = list(loss_data.keys())
loss_sens  = [v[0] for v in loss_data.values()]
loss_spec  = [v[1] for v in loss_data.values()]
colors_loss = ['steelblue', 'steelblue',
               'coral', 'coral']

x     = np.arange(len(loss_names))
width = 0.35
ax8.bar(x - width/2, loss_sens, width,
        label='Sensitivity',
        color=['lightblue', 'steelblue',
               'lightsalmon', 'coral'],
        edgecolor='white')
ax8.bar(x + width/2, loss_spec, width,
        label='Specificity',
        color=['lightblue', 'steelblue',
               'lightsalmon', 'coral'],
        alpha=0.5, edgecolor='white')
ax8.axhline(y=0.88, color='red',
            linestyle='--', alpha=0.5)
ax8.set_xticks(x)
ax8.set_xticklabels(loss_names)
ax8.set_title('CLS vs REG Loss Function', fontsize=11)
ax8.set_ylim(0, 1.1)

# ── กราฟ 9: Summary Radar Chart ──
ax9 = fig.add_subplot(gs[2, 2], polar=True)
categories = ['Sensitivity', 'Specificity',
              'AUC', 'Avg Domain\nAccuracy']
N      = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

radar_models = {
    'LLM (cut=7)'    : [0.909, 0.958, 0.944, 0.85],
    'ResNet50 REG v1': [0.886, 0.616, 0.838, 0.65],
    'ResNet50 CLS v1': [0.778, 0.843, 0.870, 0.65],
    'ViT REG NoRot'  : [0.850, 0.708, 0.868, 0.65],
}
radar_colors = ['gold', 'red', 'blue', 'green']

for (name, values), color in zip(
    radar_models.items(), radar_colors
):
    vals = values + values[:1]
    ax9.plot(angles, vals, 'o-', linewidth=2,
             label=name, color=color)
    ax9.fill(angles, vals, alpha=0.1, color=color)

ax9.set_xticks(angles[:-1])
ax9.set_xticklabels(categories, fontsize=8)
ax9.set_ylim(0, 1)
ax9.set_title('Model Comparison\nRadar Chart',
              fontsize=11, pad=20)
ax9.legend(loc='upper right',
           bbox_to_anchor=(1.4, 1.1), fontsize=7)

# # ── กราฟ 10: Improvement Roadmap ──
# ax10 = fig.add_subplot(gs[3, :])
# ax10.axis('off')
# ax10.text(0.5, 0.95,
#           'Improvement Roadmap & Insights',
#           transform=ax10.transAxes,
#           ha='center', va='top',
#           fontsize=14, fontweight='bold')

# ax10.text(0.05, 0.85, insights_text,
#           transform=ax10.transAxes,
#           ha='left', va='top',
#           fontsize=10,
#           fontfamily='monospace',
#           bbox=dict(boxstyle='round',
#                     facecolor='lightyellow',
#                     alpha=0.8))

plt.suptitle(
    'CDT Dementia Screening — Complete Evaluation Report\n'
    'ResNet50 / ViT vs LLM Baseline',
    fontsize=16, fontweight='bold', y=1.01
)

plt.savefig(
    f'{EVAL_FOLDER}/complete_evaluation.png',
    dpi=150, bbox_inches='tight'
)
plt.show()

In [ ]:
# Cell 7.5 — เพิ่มหลัง Cell 7
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

model_keys  = ['resnet50_reg', 'resnet50_cls',
               'vit_reg', 'vit_cls']
short_names = ['ResNet\nREG', 'ResNet\nCLS',
               'ViT\nREG', 'ViT\nCLS']

# ผล original
orig_auc = [0.838, 0.870, 0.866, 0.859]
orig_sens = [0.886, 0.778, 0.719, 0.713]
orig_spec = [0.616, 0.843, 0.875, 0.907]

# ผล Fix+NoRot (A)
a_auc  = [fix_A.get(k, {}).get('auc', 0)
           for k in model_keys]
a_sens = [fix_A.get(k, {}).get('sensitivity', 0)
          for k in model_keys]
a_spec = [fix_A.get(k, {}).get('specificity', 0)
          for k in model_keys]

# ผล Fix+Rot+TTA (B) cutoff=6
b_auc  = [fix_B.get(k, {}).get('6', {}).get('auc', 0)
          for k in model_keys]
b_sens = [fix_B.get(k, {}).get('6', {}).get(
    'sensitivity', 0) for k in model_keys]
b_spec = [fix_B.get(k, {}).get('6', {}).get(
    'specificity', 0) for k in model_keys]

x     = np.arange(len(model_keys))
width = 0.25

# กราฟ 1: AUC
for vals, label, color in [
    (orig_auc, 'Original',       'steelblue'),
    (a_auc,    'A: Fix+NoRot',   'orange'),
    (b_auc,    'B: Fix+Rot+TTA', 'green'),
]:
    offset = [-width, 0, width][
        ['Original', 'A: Fix+NoRot',
         'B: Fix+Rot+TTA'].index(label)
    ]
    bars = axes[0].bar(
        x + offset, vals, width,
        label=label, color=color,
        edgecolor='white'
    )

axes[0].axhline(y=0.91, color='red',
                linestyle='--', label='Target')
axes[0].set_xticks(x)
axes[0].set_xticklabels(short_names)
axes[0].set_title('AUC-ROC\n(Primary Metric)',
                  fontsize=12)
axes[0].set_ylim(0.5, 1.1)
axes[0].legend(fontsize=8)

# กราฟ 2: Sensitivity
for vals, label, color in [
    (orig_sens, 'Original',       'steelblue'),
    (a_sens,    'A: Fix+NoRot',   'orange'),
    (b_sens,    'B: Fix+Rot+TTA', 'green'),
]:
    offset = [-width, 0, width][
        ['Original', 'A: Fix+NoRot',
         'B: Fix+Rot+TTA'].index(label)
    ]
    axes[1].bar(
        x + offset, vals, width,
        label=label, color=color,
        edgecolor='white'
    )

axes[1].axhline(y=0.88, color='red',
                linestyle='--', label='Target')
axes[1].set_xticks(x)
axes[1].set_xticklabels(short_names)
axes[1].set_title('Sensitivity', fontsize=12)
axes[1].set_ylim(0, 1.1)
axes[1].legend(fontsize=8)

# กราฟ 3: Specificity
for vals, label, color in [
    (orig_spec, 'Original',       'steelblue'),
    (a_spec,    'A: Fix+NoRot',   'orange'),
    (b_spec,    'B: Fix+Rot+TTA', 'green'),
]:
    offset = [-width, 0, width][
        ['Original', 'A: Fix+NoRot',
         'B: Fix+Rot+TTA'].index(label)
    ]
    axes[2].bar(
        x + offset, vals, width,
        label=label, color=color,
        edgecolor='white'
    )

axes[2].axhline(y=0.82, color='red',
                linestyle='--', label='Target')
axes[2].set_xticks(x)
axes[2].set_xticklabels(short_names)
axes[2].set_title('Specificity', fontsize=12)
axes[2].set_ylim(0, 1.1)
axes[2].legend(fontsize=8)

plt.suptitle(
    'Fix Rotation Impact\n'
    'Original vs A (Fix+NoRot) vs B (Fix+Rot+TTA)',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    f'{EVAL_FOLDER}/fix_rotation_comparison.png',
    dpi=150, bbox_inches='tight'
)
plt.show()

In [ ]:
print('='*70)
print('FINAL RECOMMENDATION')
print('='*70)

print('''
┌──────────────────────────────────────────────────────────────────┐
│  คำถาม                  │  คำตอบ               │  เหตุผล         │
├──────────────────────────────────────────────────────────────────┤
│  ควรใช้ model ไหน?       │  LLM (Claude API)    │  ดีกว่าทุกอย่าง     │
│  ถ้าต้อง deploy offline   │  ResNet50 + REG      │  Sens=0.886 ✅  │
├──────────────────────────────────────────────────────────────────┤
│  loss function ไหน?     │  Regression (MSE)    │  ดีกว่า CLS       │
│                         │                      │  ทุกกรณี         │
├──────────────────────────────────────────────────────────────────┤
│  augment ไหม?           │  ไม่ augment หรือ      │  Rotation ทำ   │
│                         │  augment ไม่หมุน       │  label ผิดทิศ    │
├──────────────────────────────────────────────────────────────────┤
│  ลดรูปไหม?               │  ลดรูป               │  Balance class   │
│                         │  (undersample)      │  ดีกว่า            │
├──────────────────────────────────────────────────────────────────┤
│  domain ไหนต้องแก้?      │  B: worst_quadrant   │  Class 2        │
│                         │                     │  acc=0.034!      │
├──────────────────────────────────────────────────────────────────┤
│  cutoff ควรเป็นเท่าไหร่?   │  6 (มาตรฐาน CCSS)    │  หรือ 7 สำหรับ     │
│                         │                     │  LLM baseline    │
└──────────────────────────────────────────────────────────────────┘

สรุปโดยรวม:
ระบบที่ดีที่สุดสำหรับ deploy คือ
LLM (Claude API) + cutoff=7
→ Sensitivity=0.909
→ Specificity=0.958
→ AUC=0.944

ข้อจำกัด: Privacy concern + ค่าใช้จ่าย API

ถ้าต้องการ local deployment:
ResNet50 + MSE + ไม่ augment/augment ไม่หมุน
→ Sensitivity=0.886
→ ต้องแก้ Specificity ด้วย Focal Loss
''')

summary = {
    'best_model'       : 'LLM (cutoff=7)',
    'best_dl_model'    : 'ResNet50 + MSE',
    'best_loss'        : 'Regression (MSE)',
    'best_augmentation': 'No Augment or No Rotation',
    'hardest_domain'   : 'B: worst_quadrant',
    'key_findings'     : [
        'LLM baseline ดีที่สุดแต่มี circular evaluation problem',
        'ResNet50 REG Sensitivity=0.886 ผ่านเกณฑ์',
        'Regression loss ดีกว่า Classification ทุกกรณี',
        'Rotation augment ทำให้แย่ลงเพราะเข็มนาฬิกาผิดทิศ',
        'Domain B Class 2 accuracy=0.034 แย่ที่สุด',
        'Model bias ไปทาง Class 1 ทุก domain',
    ]
}

with open(f'{EVAL_FOLDER}/final_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(f'  {EVAL_FOLDER}/complete_evaluation.png')
print(f'  {EVAL_FOLDER}/final_summary.json')

In [ ]:
insights_text = '''
Short-term (ทำได้เลย):
  1. ใช้ Focal Loss แทน CrossEntropy → แก้ class imbalance ใน 0/1/2
  2. เพิ่ม class weight per class (ไม่ใช่แค่ dementia/normal)
  3. Ensemble LLM + ResNet50 REG → ได้ทั้ง Sensitivity และ Specificity

Medium-term:
  4. Human expert label โดยเฉพาะ Domain B (worst_quadrant)
  5. Fix rotation ด้วย OpenCV ก่อน train
  6. Crop เฉพาะ quadrant สำหรับ Domain B

Long-term:
  7. เก็บ data เพิ่มอีก 10x (ต้องการ ~6,000 dementia รูป)
  8. ใช้ Vision-Language Model ที่ fine-tune กับ CDT โดยเฉพาะ
  9. Clinical validation กับ human expert labels
'''